**Install & Imports**

In [39]:
!pip -q install rapidfuzz

import os, re, json
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz


**Load the USDA dataset (train.csv)**

In [40]:
USDA_DIR = "/kaggle/input/usda-national-nutrient-database"  
usda_df = pd.read_csv(os.path.join(USDA_DIR, "train.csv"))

# ---- HARD-SET correct columns  ----
NAME_COL  = "Descrip"
KCAL_COL  = "Energy_kcal"
PROT_COL  = "Protein_g"
FAT_COL   = "Fat_g"
CARB_COL  = "Carb_g"
SUGAR_COL = "Sugar_g"
FIBER_COL = "Fiber_g"

print("Loaded USDA rows:", len(usda_df))
print("Example names:", usda_df[NAME_COL].head(5).tolist())


Loaded USDA rows: 6894
Example names: ['Soy flour, full-fat, roasted', 'Pie, coconut custard, commercially prepared', 'Fish, tilapia, raw', 'Cereals, QUAKER, Instant Oatmeal, Banana Bread, dry', 'Cornmeal, degermed, enriched, yellow']


In [41]:
print(usda_df[[NAME_COL]].head(10))
print(usda_df.head(3))


                                             Descrip
0                       Soy flour, full-fat, roasted
1        Pie, coconut custard, commercially prepared
2                                 Fish, tilapia, raw
3  Cereals, QUAKER, Instant Oatmeal, Banana Bread...
4               Cornmeal, degermed, enriched, yellow
5                                Ice creams, vanilla
6                    Frozen novelties, ice type, pop
7                               Butter, without salt
8  Figs, canned, light syrup pack, solids and liq...
9  Dairy drink mix, chocolate, reduced calorie, w...
      ID                       FoodGroup  \
0  16116     Legumes and Legume Products   
1  18316                  Baked Products   
2  15261  Finfish and Shellfish Products   

                                       Descrip  Energy_kcal  Protein_g  Fat_g  \
0                 Soy flour, full-fat, roasted        441.0      34.80  21.86   
1  Pie, coconut custard, commercially prepared        260.0       5.90  13.20 

**Load Phase 3 portions**

In [42]:
PHASE3_JSON = "/kaggle/input/phase3-output/phase3_portions.json"  

with open(PHASE3_JSON, "r") as f:
    phase3 = json.load(f)

portions = phase3["portions"] if isinstance(phase3, dict) and "portions" in phase3 else phase3

# Remove background (important)
portions = [p for p in portions if str(p.get("label","")).lower() not in ["background", "bg"]]
print("Loaded portions:", len(portions))
print("Example portion:", portions[0] if portions else None)



Loaded portions: 4
Example portion: {'label': 'carrot', 'confidence': 0.9548707008361816, 'area_fraction': 0.4605979919433594, 'portion_score': 0.4605979919433594, 'estimated_grams': 644.8371887207031}


**normalization + matcher index**

In [43]:
def normalize_text(s: str) -> str:
    s = str(s).lower().strip()
    s = s.replace("_", " ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

usda_df["desc_norm"] = usda_df[NAME_COL].map(normalize_text)
choices = usda_df["desc_norm"].tolist()

# Helpful overrides for labels -> likely USDA text
OVERRIDES = {
    "french beans": "beans snap green",
    "french bean": "beans snap green",
    "green beans": "beans snap green",
    "string beans": "beans snap green",
    "shrimp": "shrimp",
    "prawn": "shrimp",
    "rice": "rice cooked",
    "carrot": "carrots raw",

}

MIN_MATCH_SCORE = 60   # lowered so rice etc can match
CONF_THRESH = 0.30

**Match + extract per-100g nutrition**

In [44]:
def match_usda(label):
    q = normalize_text(label)
    q = OVERRIDES.get(q, q)

    m = process.extractOne(q, choices, scorer=fuzz.WRatio)
    if m is None:
        return None, 0

    matched_norm, score, idx = m
    if score < MIN_MATCH_SCORE:
        return None, score

    row = usda_df.iloc[idx]

    def fcol(col):
        v = row[col]
        return float(v) if pd.notna(v) else None

    per100 = {
        "matched_name": row[NAME_COL],
        "kcal": fcol(KCAL_COL),
        "protein_g": fcol(PROT_COL),
        "fat_g": fcol(FAT_COL),
        "carbs_g": fcol(CARB_COL),
        "sugar_g": fcol(SUGAR_COL),
        "fiber_g": fcol(FIBER_COL),
    }
    # apply minimum confidence threshold
    if score < 90:
        return None, score
        
    return per100, score

# grams per 1% of plate area (heuristic, cited as assumption)
GRAMS_PER_1PCT = {
    "rice": 12,        # fluffy but dense
    "bean": 8,         # vegetables
    "beans": 8,
    "carrot": 6,       # light vegetable
    "vegetable": 6,
    "shrimp": 18,      # protein, denser
    "fish": 18,
    "meat": 20,
    "default": 10
}

def recompute_grams_from_area(p):
    label = str(p["label"]).lower()
    area_frac = float(p.get("area_fraction", 0.0))

    g_per_pct = GRAMS_PER_1PCT["default"]
    for k, v in GRAMS_PER_1PCT.items():
        if k != "default" and k in label:
            g_per_pct = v
            break
    return float(g_per_pct * (area_frac * 100.0))

def get_grams(p):
    # try phase-3 estimated grams if present
    eg = p.get("estimated_grams", None)
    if eg is not None:
        eg = float(eg)

        # reject unrealistic portions
        # (you can tune these caps)
        if eg < 5 or eg > 400:   # >400g is suspicious for a single item like carrot
            return recompute_grams_from_area(p)
        return eg

    return recompute_grams_from_area(p)


def scale(per100, grams):
    factor = grams / 100.0
    return {
        k: (round(per100[k] * factor, 2) if per100[k] is not None else None)
        for k in ["kcal","protein_g","fat_g","carbs_g","sugar_g","fiber_g"]
    }

**Run Phase 4 nutrition mapping**

In [45]:
items = []
totals = {"kcal":0,"protein_g":0,"fat_g":0,"carbs_g":0,"sugar_g":0,"fiber_g":0}

for p in portions:
    if float(p.get("confidence", 1.0)) < CONF_THRESH:
        continue

    grams = get_grams(p)
    per100, score = match_usda(p["label"])


    if per100 is None:
        items.append({
            "label": p["label"],
            "confidence": float(p.get("confidence", 1.0)),
            "grams": grams,
            "match_status": "no_match",
            "match_score": score
        })
        continue

    scaled = scale(per100, grams)

    items.append({
        "label": p["label"],
        "confidence": float(p.get("confidence", 1.0)),
        "grams": grams,
        "usda_match": per100["matched_name"],
        "match_score": score,
        "per100g": {k:v for k,v in per100.items() if k != "matched_name"},
        "nutrition": scaled
    })

    for k in totals:
        if scaled.get(k) is not None:
            totals[k] += float(scaled[k])
            # round totals to 2 decimal places
            totals[k] = round(totals[k], 2)

print("\nMapped items:", len(items))
print("Totals:", totals)

for it in items:
    if "nutrition" in it:
        print(f"{it['label']:12s} grams={it['grams']:.1f} kcal={it['nutrition']['kcal']:.1f}")



Mapped items: 4
Totals: {'kcal': 440.53, 'protein_g': 29.34, 'fat_g': 2.48, 'carbs_g': 78.79, 'sugar_g': 18.45, 'fiber_g': 15.11}
carrot       grams=276.4 kcal=96.7
rice         grams=225.9 kcal=228.2
French beans grams=112.1 kcal=34.7
shrimp       grams=80.9 kcal=80.9


In [46]:
for it in items:
    if it.get("match_status") == "no_match":
        continue
    print(f"{it['label']:12s} grams={it['grams']:.1f}  match={it['usda_match']}")


carrot       grams=276.4  match=Carrots, baby, raw
rice         grams=225.9  match=Wild rice, cooked
French beans grams=112.1  match=Beans, snap, green, raw
shrimp       grams=80.9  match=Crustaceans, shrimp, mixed species, canned


**Save Phase 4 output JSON**

In [47]:
GRAMS_PER_1PCT_DEFAULT = GRAMS_PER_1PCT.get("default", 10)


phase4 = {
    "method": "usda_traincsv_fuzzy_match_scaled_by_grams",
    "settings": {
        "min_match_score": MIN_MATCH_SCORE,
        "confidence_threshold": CONF_THRESH,
        "grams_per_1pct_default": GRAMS_PER_1PCT_DEFAULT,
        "name_col": NAME_COL
    },
    "items": items,
    "totals": totals
}

OUT_PATH = "/kaggle/working/phase4_nutrition.json"
with open(OUT_PATH, "w") as f:
    json.dump(phase4, f, indent=2)

print("\n✅ Saved:", OUT_PATH)


✅ Saved: /kaggle/working/phase4_nutrition.json


In [51]:
# ============================================
# PHASE 4 TESTING
# ============================================

print("=== PHASE 4 TESTING ===")

# -------------------------------
# Test 1: Output structure
# -------------------------------
assert isinstance(phase4, dict), "phase4 output should be a dictionary"
assert "items" in phase4, "phase4 must contain 'items'"
assert "totals" in phase4, "phase4 must contain 'totals'"
assert isinstance(phase4["items"], list), "'items' should be a list"
assert isinstance(phase4["totals"], dict), "'totals' should be a dictionary"
print("Test 1 passed: Output structure is correct")


# -------------------------------
# Test 2: Totals keys exist
# -------------------------------
expected_total_keys = ["kcal", "protein_g", "fat_g", "carbs_g", "sugar_g", "fiber_g"]
for k in expected_total_keys:
    assert k in phase4["totals"], f"Missing total key: {k}"
print("Test 2 passed: Totals contain all required nutrition fields")


# -------------------------------
# Test 3: Non-negative totals
# -------------------------------
for k, v in phase4["totals"].items():
    assert v >= 0, f"Total {k} is negative"
print("Test 3 passed: Totals are non-negative")


# -------------------------------
# Test 4: Item-level structure
# -------------------------------
for i, item in enumerate(phase4["items"]):
    assert "label" in item, f"Item {i} missing label"
    assert "grams" in item, f"Item {i} missing grams"
    assert item["grams"] >= 0, f"Item {i} has negative grams"
print("Test 4 passed: Each item has valid basic fields")


# -------------------------------
# Test 5: Confidence range
# -------------------------------
for i, p in enumerate(portions):
    conf = float(p.get("confidence", 1.0))
    assert 0.0 <= conf <= 1.0, f"Portion {i} confidence out of range: {conf}"
print("Test 5 passed: Confidence values are within [0,1]")


# -------------------------------
# Test 6: Area fraction range
# -------------------------------
for i, p in enumerate(portions):
    af = float(p.get("area_fraction", 0.0))
    assert 0.0 <= af <= 1.0, f"Portion {i} area_fraction out of range: {af}"
print("Test 6 passed: Area fractions are within [0,1]")


# -------------------------------
# Test 7: Match function works
# -------------------------------
sample_labels = ["rice", "shrimp", "carrot", "unknown_food_xyz"]

for lbl in sample_labels:
    per100, score = match_usda(lbl)
    
    if per100 is None:
        print(f"Label: {lbl:15s} | Match score: {score} | NO MATCH")
    else:
        print(f"Label: {lbl:15s} | Match score: {score} | MATCHED")

print("Test 7 completed: USDA matching checked")


# -------------------------------
# Test 8: Gram fallback logic
# -------------------------------
test_portion = {"label": "rice", "area_fraction": 0.25, "confidence": 0.9}
g = get_grams(test_portion)
assert g >= 0, "Fallback grams should be non-negative"
print("Test 8 passed: Gram fallback works")


# -------------------------------
# Test 9: Unrealistic estimated grams fallback
# -------------------------------
test_portion2 = {"label": "carrot", "area_fraction": 0.10, "estimated_grams": 1000, "confidence": 0.9}
g2 = get_grams(test_portion2)
assert g2 < 1000, "Unrealistic estimated grams should trigger fallback"
print("Test 9 passed: Unrealistic grams fallback works")


# -------------------------------
# Test 10: Scaled nutrition non-negative
# -------------------------------
for item in phase4["items"]:
    if item.get("match_status") == "no_match":
        continue
    for k in ["kcal", "protein_g", "fat_g", "carbs_g", "sugar_g", "fiber_g"]:
        if item.get(k) is not None:
            assert item[k] >= 0, f"{k} is negative for item {item['label']}"
print("Test 10 passed: Scaled nutrition values are non-negative")

print("\n✅ All Phase 4 tests completed successfully")

=== PHASE 4 TESTING ===
Test 1 passed: Output structure is correct
Test 2 passed: Totals contain all required nutrition fields
Test 3 passed: Totals are non-negative
Test 4 passed: Each item has valid basic fields
Test 5 passed: Confidence values are within [0,1]
Test 6 passed: Area fractions are within [0,1]
Label: rice            | Match score: 95.0 | MATCHED
Label: shrimp          | Match score: 90.0 | MATCHED
Label: carrot          | Match score: 95.0 | MATCHED
Label: unknown_food_xyz | Match score: 85.5 | NO MATCH
Test 7 completed: USDA matching checked
Test 8 passed: Gram fallback works
Test 9 passed: Unrealistic grams fallback works
Test 10 passed: Scaled nutrition values are non-negative

✅ All Phase 4 tests completed successfully


In [49]:
# ============================================
# SIMPLE EVALUATION AGAINST EXPECTED VALUES
# ============================================

print("=== PHASE 4 EVALUATION ===")

# Example manually checked expectations
# Replace these with values you calculate from USDA manually for your chosen test foods
ground_truth = [
    {"label": "rice", "expected_kcal": 130},
    {"label": "carrot", "expected_kcal": 41},
    {"label": "shrimp", "expected_kcal": 99},
]

results = []

for gt in ground_truth:
    per100, score = match_usda(gt["label"])
    if per100 is None:
        print(f"No match for {gt['label']}")
        continue

    pred_kcal = per100["kcal"]
    abs_error = abs(pred_kcal - gt["expected_kcal"])

    results.append(abs_error)
    print(f"{gt['label']:10s} | Pred kcal={pred_kcal:.2f} | Expected={gt['expected_kcal']:.2f} | Abs Error={abs_error:.2f}")

if results:
    mae = np.mean(results)
    rmse = np.sqrt(np.mean(np.square(results)))
    print(f"\nMAE  = {mae:.2f}")
    print(f"RMSE = {rmse:.2f}")

=== PHASE 4 EVALUATION ===
rice       | Pred kcal=101.00 | Expected=130.00 | Abs Error=29.00
carrot     | Pred kcal=35.00 | Expected=41.00 | Abs Error=6.00
shrimp     | Pred kcal=100.00 | Expected=99.00 | Abs Error=1.00

MAE  = 12.00
RMSE = 17.11
